In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
from diffusion_geometry.visualisation import *
from plotly.subplots import make_subplots
from diffusion_geometry import Function, VectorField, Form
from generate_data import gen_2d_data
from diffusion_geometry import DiffusionGeometry

In [ ]:
def sample_annulus(center=(0.0, 0.0), r_inner=0.7, r_outer=1.0, n=500, jitter=0.02, noise_level=0.005):
    # Draw radius uniformly in *area*, not linearly — so density is uniform
    u = np.random.rand(n)
    radii = np.sqrt(r_inner**2 + (r_outer**2 - r_inner**2) * u)

    # Angles with slight random jitter
    theta = 2 * np.pi * (np.arange(n) / n + np.random.uniform(-jitter, jitter, n))
    np.random.shuffle(theta)  # break angular ordering (prevents rings)

    # Convert to Cartesian coordinates
    x = radii * np.cos(theta) + center[0]
    y = radii * np.sin(theta) + center[1]

    # Optional small isotropic noise to avoid perfect circular symmetry
    noise = np.random.normal(scale=noise_level * (r_outer - r_inner), size=(n, 2))
    data = np.column_stack((x, y)) + noise
    data *= [1.3, 1]
    return data


def generate_function_function_figures(fig, current_row):
    data = sample_annulus(noise_level=0.5)
    
    dg = DiffusionGeometry.from_point_cloud(data)
    n_function_basis, n_coefficients = dg.n_function_basis, dg.n_coefficients

    f1 = Function.from_pointwise_basis(np.zeros(dg.n), dg)
    f1.coeffs[1] = 1 

    f2 = Function.from_pointwise_basis(np.zeros(dg.n), dg)
    f2.coeffs[2] = 1


    fig1 = plot_scatter_2d(data, color=f1.to_ambient(), size=3)
    fig2 = plot_scatter_2d(data, color=f2.to_ambient(), size=3)
    fig3 = plot_scatter_2d(data, color=(f1*f2).to_ambient(), size=3)

    fig.add_traces(list(fig1.data), rows=current_row, cols=1)
    fig.add_traces(list(fig2.data), rows=current_row, cols=2)
    fig.add_traces(list(fig3.data), rows=current_row, cols=3)



def generate_function_1form_figure(fig, current_row):
    data = sample_annulus(center=(0,0), r_inner=0.5, r_outer=1.0, n=300, noise_level=0.1)

    dg = DiffusionGeometry.from_point_cloud(data)
    n_function_basis, n_coefficients = dg.n_function_basis, dg.n_coefficients

    f_coeffs = np.zeros(n_function_basis)
    f_coeffs[3] = 1
    f = Function.from_coeffs(f_coeffs, dg)

    radii = np.linalg.norm(data, axis=1)
    angles = np.arctan2(data[:,1], data[:,0]) + np.pi / 2
    vf_data = np.vstack((np.cos(angles), np.sin(angles))).T
    # vf_data = vf_data * radii[:, np.newaxis]  # Scale by radius to get circular field
    rot_vf = VectorField.from_pointwise_basis(vf_data, dg)

    quiver_scale = 50
    arrow_scale = 0.3
    line_width = 1

    fig1 = plot_scatter_2d(data, color=f.to_ambient(), size=3)
    fig2 = plot_quiver_2d(data, rot_vf.to_ambient(), scale=quiver_scale, arrow_scale=arrow_scale, line_width=line_width)
    fig3 = plot_quiver_2d(data, (f*rot_vf.flat()).to_ambient(), scale=quiver_scale, arrow_scale=arrow_scale, line_width=line_width)

    fig.add_traces(list(fig1.data), rows=current_row, cols=1)
    fig.add_traces(list(fig2.data), rows=current_row, cols=2)
    fig.add_traces(list(fig3.data), rows=current_row, cols=3)


def sample_sphere(n_points=1000, seed = 1):
    np.random.seed(seed)
    phi = np.random.uniform(0, 2*np.pi, n_points)
    cos_theta = np.random.uniform(-1, 1, n_points)
    theta = np.arccos(cos_theta)
    x = np.sin(theta) * np.cos(phi)
    y = np.sin(theta) * np.sin(phi)
    z = np.cos(theta)
    return np.column_stack((x, y, z))



def generate_function_2form_figure(fig, current_row):
    n = 2000
    data = sample_sphere(n)
    dg = DiffusionGeometry.from_point_cloud(
        data,
    )

    # vals, vecs = dg.laplacian(2).spectrum()
    # volume_form = vecs[0]

    volume_form_data = np.zeros((dg.n,3))
    volume_form_data[:,0] = data[:, 2]
    volume_form_data[:,1] = -data[:, 1]
    volume_form_data[:,2] = data[:, 0]


    volume_form = 0.1 * Form.from_pointwise_basis(volume_form_data, dg, 2)

    n_circle = 20

    fig1 = plot_2form_3d(data, volume_form.to_ambient(), n_circle=n_circle, radius=5e+4)
   
    scaling_function = Function.from_pointwise_basis(data[:,0], dg)

    fig2 = plot_scatter_3d(data, color=scaling_function.to_ambient(), size=1.5)

    fig3 = plot_2form_3d(data, (scaling_function*volume_form).to_ambient(), n_circle=n_circle, radius=5e+4)


    fig.add_traces(list(fig2.data), rows=current_row, cols=1)
    fig.add_traces(list(fig1.data), rows=current_row, cols=2)
    fig.add_traces(list(fig3.data), rows=current_row, cols=3)



def genereate_1form_1form_figure(fig, current_row):
    r=1
    n_side = 28
    x = np.linspace(-r, r, n_side)
    y = np.linspace(-r, r, n_side)
    X, Y = np.meshgrid(x, y)
    data = np.column_stack((X.ravel(), Y.ravel())) * [1.3, 1]
    data += np.random.randn(data.shape[0], 2) * 0.02
   

    dg = DiffusionGeometry.from_point_cloud(data)

    n_function_basis, n_coefficients = dg.n_function_basis, dg.n_coefficients

    quiver_scale = 13
    arrow_scale = 0.5
    line_width = 1

    def f(x,y):
        return np.sin(5*x)*np.cos(4*x)
    def g(x,y):
        return np.cos(3*y)*np.sin(2*y)


    f_data = f(data[:,0], data[:,1])
    g_data = g(data[:,0], data[:,1])

    f = Function.from_pointwise_basis(f_data, dg)
    g = Function.from_pointwise_basis(g_data, dg)

    fig1 = plot_quiver_2d(data, (f.d().sharp()).to_ambient(), scale=quiver_scale, arrow_scale=arrow_scale, line_width=line_width)
    fig2 = plot_quiver_2d(data, (g.d().sharp()).to_ambient(), scale=quiver_scale, arrow_scale=arrow_scale, line_width=line_width)
    fig3 = plot_2form_2d(data, ((f.d() ^ g.d())).to_ambient(), n_circle=dg.n)

    fig.add_traces(list(fig1.data), rows=current_row, cols=1)
    fig.add_traces(list(fig2.data), rows=current_row, cols=2)
    fig.add_traces(list(fig3.data), rows=current_row, cols=3)


def generate_1form_1form_3d(fig, current_row):
    n= 2200

    data = sample_sphere(n)
    dg = DiffusionGeometry.from_point_cloud(
        data
    )

    f_data = np.zeros(dg.n)
    f_data[:] = np.sin(2*data[:, 0])  # x-coordinate function

    f = Function.from_pointwise_basis(f_data, dg)
    df = f.d()

    f2_data = np.zeros(dg.n)
    f2_data[:] = np.cos(2*data[:, 1])  # y-coordinate
    f2 = Function.from_pointwise_basis(f2_data, dg)
    df2 = f2.d()

    arrow_scale = 1.4
    line_width = 1.0
    scale = 20

    n_circle = 20

    fig1 = plot_quiver_3d(data, df.sharp().to_ambient(), scale=scale, arrow_scale=arrow_scale, line_width=line_width)
    fig2 = plot_quiver_3d(data, df2.sharp().to_ambient(), scale=scale, arrow_scale=0.5*arrow_scale, line_width=line_width)
    fig3 = plot_2form_3d(data, (df ^ df2).to_ambient(), n_circle=n_circle, radius=1200)

    fig.add_traces(list(fig1.data), rows=current_row, cols=1)
    fig.add_traces(list(fig2.data), rows=current_row, cols=2)
    fig.add_traces(list(fig3.data), rows=current_row, cols=3)


num_rows = 5
num_cols = 3

specs = specs = [
    [{"type": "xy"}]*num_cols,  
    [{"type": "xy"}]*num_cols,    
    [{"type": "scene"}]*num_cols,  
    [{"type": "xy"}]*num_cols,  
    [{"type": "scene"}]*num_cols,  
]

fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.01,
    shared_xaxes=True,
    shared_yaxes=True
)

generate_function_function_figures(fig, current_row=1)
generate_function_1form_figure(fig, current_row=2)
generate_function_2form_figure(fig, current_row=3)
genereate_1form_1form_figure(fig, current_row=4)
generate_1form_1form_3d(fig, current_row=5)

# Same aspect ratio for all subplots
for r in range(num_rows):
    for c in range(num_cols):
        fig.update_xaxes(scaleanchor=f"y{c+1}", scaleratio=1, row=r+1, col=c+1)
        fig.update_yaxes(scaleanchor=f"x{c+1}", scaleratio=1, row=r+1, col=c+1)


camera = dict(
    eye=dict(x=0, y=0, z=1.3),  # directly above (along +z)
    up=dict(x=0, y=1, z=0),     # y-axis points upward in the view
    center=dict(x=0, y=0, z=0)
)

axis_range = dict(range=[-1.1, 1.1])

for i in range(1, num_rows * num_cols + 1):
    scene_key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout({
        scene_key: dict(
            camera=camera,
            aspectmode="data",
            xaxis=dict(**axis_range, title='x'),
            yaxis=dict(**axis_range, title='y'),
            zaxis=dict(**axis_range, title='z'),
        )
    })

fig.update_layout(width=800, height=1000) 


clean_fig(fig)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))
fig.show()

fig.write_image("figs/products.pdf", scale=1)
